# Experiment 2: ResNet-50 at 320x320 with cosine scheduling

Controlled comparison with Experiment 1. Only the cosine scheduler, 40-epoch budget, and patience of 7 change. Test evaluation is disabled by default.

In [ ]:
import gc, json, os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists(): raise RuntimeError('Could not find src')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    classification_report, cohen_kappa_score, confusion_matrix,
    precision_recall_fscore_support)
from torch.utils.data import DataLoader
from torchvision import transforms
from src.datasets.glaucoma_dataset import GlaucomaDataset, IDX_TO_CLASS
from src.models.resnet50 import ResNet50Classifier
from src.training.config import (CHECKPOINT_DIR, NUM_CLASSES, RESULTS_FIGURES_DIR,
    RESULTS_METRICS_DIR, SEED, WEIGHT_DECAY, set_seed)

if not torch.cuda.is_available(): raise RuntimeError('CUDA is not available')
DEVICE = torch.device('cuda:0')
set_seed(SEED)
sns.set_theme(style='whitegrid')
print('Project root:', PROJECT_ROOT)
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

## Configuration

In [ ]:
EXPERIMENT_NAME = 'EXPERIMENT_2_320_COSINE'
BASELINE_NAME = 'EXPERIMENT_1_320'
IMAGE_SIZE, BATCH_SIZE = 320, 16
BATCH_SIZE_CANDIDATES = (16, 8)
AMP_ENABLED, RUN_FINAL_TEST = True, False
INITIAL_LR, MIN_LR = 1e-4, 1e-6
EPOCHS, PATIENCE = 40, 7
CLASS_NAMES = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]
CHECKPOINT_PATH = Path(CHECKPOINT_DIR) / 'resnet50_320_cosine_best.pt'

assert IMAGE_SIZE == 320 and INITIAL_LR == 1e-4 and WEIGHT_DECAY == 1e-4
assert CHECKPOINT_PATH != Path(CHECKPOINT_DIR) / 'resnet50_320_best.pt'
print({'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE, 'amp': AMP_ENABLED,
       'epochs': EPOCHS, 'patience': PATIENCE, 'initial_lr': INITIAL_LR,
       'scheduler': f'CosineAnnealingLR(T_max={EPOCHS}, eta_min={MIN_LR})',
       'checkpoint_criterion': 'lowest validation CrossEntropyLoss',
       'run_final_test': RUN_FINAL_TEST})

## Unchanged Experiment 1 data pipeline

The transforms and official train/validation/test splits are identical to Experiment 1.

In [ ]:
train_transform_320 = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform_320 = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
raw_dataset = load_dataset('moondream/glaucoma-detection')

def build_loaders(batch_size):
    train = GlaucomaDataset(raw_dataset['train'], transform=train_transform_320)
    val = GlaucomaDataset(raw_dataset['validation'], transform=eval_transform_320)
    test = GlaucomaDataset(raw_dataset['test'], transform=eval_transform_320)
    return (DataLoader(train, batch_size=batch_size, shuffle=True),
            DataLoader(val, batch_size=batch_size, shuffle=False),
            DataLoader(test, batch_size=batch_size, shuffle=False))

def verify_loaders(train_loader, val_loader, test_loader):
    assert [len(x.dataset) for x in (train_loader, val_loader, test_loader)] == [2847, 1259, 1272]
    images, labels = next(iter(train_loader))
    print('Image batch:', tuple(images.shape))
    print('Label batch:', tuple(labels.shape))
    assert images.shape[1:] == (3, 320, 320)
    assert labels.min() >= 0 and labels.max() < NUM_CLASSES

train_loader, val_loader, test_loader = build_loaders(BATCH_SIZE)
verify_loaders(train_loader, val_loader, test_loader)

## ImageNet-pretrained ResNet-50, AdamW, and cosine scheduler

In [ ]:
def build_training_objects():
    model = ResNet50Classifier(num_classes=NUM_CLASSES, pretrained=True, freeze_features=False).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
    return model, optimizer, scheduler

loss_fn = nn.CrossEntropyLoss()
probe_model, probe_optimizer, probe_scheduler = build_training_objects()
assert probe_scheduler.optimizer is probe_optimizer
assert probe_optimizer.param_groups[0]['lr'] == INITIAL_LR
print(f'Trainable parameters: {sum(p.numel() for p in probe_model.parameters() if p.requires_grad):,}')
del probe_model, probe_optimizer, probe_scheduler
torch.cuda.empty_cache()

## Training

The printed learning rate is the rate used during that epoch. The scheduler advances once after validation.

In [ ]:
def compute_metrics(labels, predictions):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=range(NUM_CLASSES), average='macro', zero_division=0)
    return {'accuracy': float(accuracy_score(labels, predictions)),
            'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)),
            'macro_precision': float(precision), 'macro_recall': float(recall),
            'macro_f1': float(f1),
            'qwk': float(cohen_kappa_score(labels, predictions, weights='quadratic'))}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, predictions, labels_all = 0.0, [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        total_loss += loss_fn(outputs, labels).item() * images.size(0)
        predictions.extend(outputs.argmax(1).cpu().tolist())
        labels_all.extend(labels.cpu().tolist())
    metrics = compute_metrics(labels_all, predictions)
    metrics['loss'] = float(total_loss / len(loader.dataset))
    return metrics, predictions, labels_all

def fit(model, optimizer, scheduler, train_loader, val_loader):
    history = {'train_loss': [], 'val_loss': [], 'val_metrics': [], 'learning_rate': []}
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_loss, best_epoch, best_metrics, best_lr, stale = float('inf'), None, None, None, 0
    for epoch in range(1, EPOCHS + 1):
        lr = optimizer.param_groups[0]['lr']
        model.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.autocast('cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                loss = loss_fn(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * images.size(0)
        train_loss = total_loss / len(train_loader.dataset)
        metrics, _, _ = evaluate(model, val_loader)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(metrics['loss'])
        history['val_metrics'].append(metrics)
        history['learning_rate'].append(lr)
        improved = metrics['loss'] < best_loss
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss {train_loss:.4f} | "
              f"Validation Loss {metrics['loss']:.4f} | Validation Accuracy {metrics['accuracy']:.4f} | "
              f"Validation Macro F1 {metrics['macro_f1']:.4f} | "
              f"Validation Balanced Accuracy {metrics['balanced_accuracy']:.4f} | "
              f"Validation QWK {metrics['qwk']:.4f} | Learning Rate {lr:.8f}"
              + (' (best validation loss)' if improved else ''))
        if improved:
            best_loss, best_epoch, best_metrics, best_lr, stale = metrics['loss'], epoch, metrics.copy(), lr, 0
            CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), CHECKPOINT_PATH)
        else:
            stale += 1
        scheduler.step()
        if stale >= PATIENCE:
            print(f'No validation-loss improvement for {PATIENCE} epochs; stopping.')
            break
    if len(history['learning_rate']) > 1: assert history['learning_rate'][-1] < history['learning_rate'][0]
    return history, best_epoch, best_metrics, best_lr

oom_occurred = False
for batch_size in BATCH_SIZE_CANDIDATES:
    gc.collect(); torch.cuda.empty_cache(); set_seed(SEED)
    train_loader, val_loader, test_loader = build_loaders(batch_size)
    print(f'\nAttempting Experiment 2 with batch size {batch_size}')
    verify_loaders(train_loader, val_loader, test_loader)
    set_seed(SEED)
    model, optimizer, scheduler = build_training_objects()
    try:
        history, best_epoch, best_val_metrics, best_learning_rate = fit(
            model, optimizer, scheduler, train_loader, val_loader)
        actual_batch_size = batch_size
        break
    except RuntimeError as error:
        if 'out of memory' not in str(error).lower(): raise
        oom_occurred = True
        print(f'CUDA OOM at batch size {batch_size}; restarting from epoch 1.')
        del model, optimizer, scheduler
else:
    raise RuntimeError('CUDA OOM at batch sizes 16 and 8')

final_learning_rate = history['learning_rate'][-1]
next_learning_rate = optimizer.param_groups[0]['lr']
assert CHECKPOINT_PATH.exists()
print({'epochs_completed': len(history['train_loss']), 'best_epoch': best_epoch,
       'best_validation_loss': best_val_metrics['loss'], 'actual_batch_size': actual_batch_size,
       'best_learning_rate': best_learning_rate, 'final_learning_rate_used': final_learning_rate,
       'next_learning_rate': next_learning_rate, 'amp': AMP_ENABLED})

## Experiment 2 curves

In [ ]:
figure_dir = Path(RESULTS_FIGURES_DIR)
figure_dir.mkdir(parents=True, exist_ok=True)
epochs_ran = list(range(1, len(history['train_loss']) + 1))
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0, 0].plot(epochs_ran, history['train_loss'], marker='o', label='Train')
axes[0, 0].plot(epochs_ran, history['val_loss'], marker='o', label='Validation')
axes[0, 0].set(title='Loss', xlabel='Epoch'); axes[0, 0].legend()
for axis, key, title in [(axes[0, 1], 'accuracy', 'Validation accuracy'),
                         (axes[1, 0], 'macro_f1', 'Validation Macro F1'),
                         (axes[1, 1], 'qwk', 'Validation QWK')]:
    axis.plot(epochs_ran, [m[key] for m in history['val_metrics']], marker='o')
    axis.set(title=title, xlabel='Epoch')
fig.suptitle('Experiment 2: ResNet-50 320x320 cosine'); fig.tight_layout()
fig.savefig(figure_dir / 'resnet50_320_cosine_training_curve.png', dpi=160)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history['learning_rate'], marker='o')
plt.xlabel('Epoch'); plt.ylabel('Learning rate used'); plt.title('Cosine learning-rate schedule')
plt.tight_layout(); plt.savefig(figure_dir / 'resnet50_320_cosine_lr_curve.png', dpi=160)
plt.show()

## Selected-checkpoint validation evaluation

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))
selected_val_metrics, val_predictions, val_labels = evaluate(model, val_loader)
print(json.dumps(selected_val_metrics, indent=2))
print(classification_report(val_labels, val_predictions, labels=range(NUM_CLASSES),
      target_names=CLASS_NAMES, digits=4, zero_division=0))

matrix = confusion_matrix(val_labels, val_predictions, labels=range(NUM_CLASSES))
plt.figure(figsize=(7, 6))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('Ground truth')
plt.title('Experiment 2 validation confusion matrix'); plt.tight_layout()
plt.savefig(figure_dir / 'resnet50_320_cosine_validation_confusion_matrix.png', dpi=160)
plt.show()

## Save results and compare Experiment 1 vs Experiment 2

In [ ]:
metrics_dir = Path(RESULTS_METRICS_DIR)
metrics_dir.mkdir(parents=True, exist_ok=True)
metrics_path = metrics_dir / 'resnet50_320_cosine.json'
test_metrics = None
if RUN_FINAL_TEST:
    test_metrics, _, _ = evaluate(model, test_loader)
else:
    print('Final test evaluation skipped (RUN_FINAL_TEST=False).')

results = {'experiment': EXPERIMENT_NAME, 'baseline': BASELINE_NAME,
 'model': 'ResNet50Classifier(pretrained=True, freeze_features=False)',
 'image_size': IMAGE_SIZE, 'batch_size': actual_batch_size, 'amp_enabled': AMP_ENABLED,
 'cuda_oom_occurred': oom_occurred, 'epochs_completed': len(history['train_loss']),
 'maximum_epochs': EPOCHS, 'early_stopping_patience': PATIENCE, 'best_epoch': best_epoch,
 'checkpoint_criterion': 'lowest validation CrossEntropyLoss', 'optimizer': 'AdamW',
 'initial_learning_rate': INITIAL_LR, 'minimum_learning_rate': MIN_LR,
 'best_learning_rate': best_learning_rate, 'final_learning_rate_used': final_learning_rate,
 'scheduler': {'name': 'CosineAnnealingLR', 'T_max': EPOCHS, 'eta_min': MIN_LR},
 'weight_decay': WEIGHT_DECAY, 'training_history': history,
 'validation_metrics': selected_val_metrics, 'test_metrics': test_metrics}
with metrics_path.open('w', encoding='utf-8') as file: json.dump(results, file, indent=2)
print('Saved:', CHECKPOINT_PATH, metrics_path, figure_dir, sep='\n')

exp1_path = metrics_dir / 'resnet50_320.json'
exp1 = {}
if exp1_path.exists():
    with exp1_path.open(encoding='utf-8') as file: exp1 = json.load(file).get('validation_metrics', {})
rows = [('Val Accuracy', 'accuracy'), ('Val Balanced Accuracy', 'balanced_accuracy'),
        ('Val Macro F1', 'macro_f1'), ('Val QWK', 'qwk'), ('Best Val Loss', 'loss')]
print('\nMetric'.ljust(26), BASELINE_NAME.rjust(20), EXPERIMENT_NAME.rjust(26))
print('-' * 74)
for label, key in rows:
    old = f"{exp1[key]:.4f}" if key in exp1 else 'not available'
    print(label.ljust(26), old.rjust(20), f"{selected_val_metrics[key]:.4f}".rjust(26))
if not exp1: print('Experiment 1 validation metrics unavailable; test metrics were not substituted.')